# YOLOv8 backbones on CB55: stock controls vs tuned PVT-v2-B2

Three arms use a YOLOv8 detection model:

1. **CSPDarknet** — the stock backbone from `yolov8.yaml`;
2. **ResNet-34** — `configs/detection/yolov8-resnet34.yaml`;
3. **PVT-v2-B2** — `configs/detection/yolov8-pvt_v2.yaml` (`pvt_v2_b2.in1k` in timm).

All arms use YOLOv8 scale `n`, explicit `optimizer="AdamW"`, input size `1024`,
and the same three seeds `(0, 1, 2)`. CSPDarknet and ResNet-34 remain stock-recipe
controls. PVT-v2-B2 gets a convergence recipe: 500 epochs, `lr0=1e-3`, cosine LR,
10 warmup epochs, and stock patience 100. Other Ultralytics defaults are unchanged.
Run the notebook top-to-bottom from `50_modelling/02_2stage`.

> Initialization note: the CSPDarknet arm is created from `yolov8n.yaml` (random
> initialization), while the existing ResNet-34 and PVT-v2-B2 YAML modules request
> their ImageNet pretrained backbone weights. No COCO detector checkpoint is loaded
> for any arm.

## Shared experiment configuration

In [ ]:
from pathlib import Path

SEEDS = (0, 1, 2)
SCALE = "n"
IMGSZ = 1024  # divisible by the detectors' maximum stride (32)
DATA_CFG = Path("configs/data/diva_cb55_detect.yaml")
PROJECT = Path("../../80_models/02_2stage/diva-hisdb/detection").resolve()
RESULTS_CSV = PROJECT / f"yolov8_backbone_stock_vs_tuned_pvt_cb55_{IMGSZ}_results.csv"

# The n-inserted local names are Ultralytics scale aliases: for example,
# yolov8n-resnet34.yaml resolves to yolov8-resnet34.yaml with scale='n'.
ARMS = {
    "cspdarknet": f"yolov8{SCALE}.yaml",
    "resnet34": Path(f"configs/detection/yolov8{SCALE}-resnet34.yaml"),
    "pvt_v2_b2": Path(f"configs/detection/yolov8{SCALE}-pvt_v2.yaml"),
}

RECIPES = {
    # Empty train_args means installed Ultralytics defaults. These tags preserve the
    # existing 1024 control run names, so completed CSP/ResNet runs are reused.
    "cspdarknet": {"tag": "stock", "train_args": {}},
    "resnet34": {"tag": "stock", "train_args": {}},
    "pvt_v2_b2": {
        "tag": "pvt500e_lr1e-3_cos_warm10",
        "train_args": {
            "epochs": 500,
            "lr0": 1e-3,
            "cos_lr": True,
            "warmup_epochs": 10.0,
        },
    },
}

assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
assert set(ARMS) == set(RECIPES)
assert DATA_CFG.is_file(), DATA_CFG
PROJECT.mkdir(parents=True, exist_ok=True)

## Imports, paths, and stock-recipe audit

The first table is read directly from the installed Ultralytics defaults. The second
shows the effective recipe for each arm. AdamW and `imgsz=1024` are shared; only the
PVT convergence arm overrides epochs, initial LR, scheduler, and warmup.

In [ ]:
import gc
import sys

import pandas as pd
import torch
from ultralytics import YOLO, __version__ as ultralytics_version
from ultralytics.cfg import DEFAULT_CFG_DICT

sys.path.insert(0, str(Path.cwd()))
import timm_backbone
import train_detector

timm_backbone.register()  # makes TimmBackbone visible to the YAML parser/checkpoints
DATA = train_detector.resolve_data(DATA_CFG)

stock_keys = [
    "epochs", "patience", "batch", "imgsz", "lr0", "lrf",
    "cos_lr", "weight_decay", "warmup_epochs", "box", "cls", "dfl",
    "hsv_h", "hsv_s", "hsv_v", "translate", "scale", "fliplr", "mosaic",
]
stock_recipe = {key: DEFAULT_CFG_DICT[key] for key in stock_keys}

print("Ultralytics:", ultralytics_version)
print("device     :", "cuda" if torch.cuda.is_available() else "cpu")
print("data       :", DATA_CFG.resolve())
print("project    :", PROJECT)
print("seeds      :", SEEDS)
display(pd.Series(stock_recipe, name="stock value").to_frame())

recipe_rows = []
for backbone, recipe in RECIPES.items():
    effective = {**stock_recipe, **recipe["train_args"]}
    recipe_rows.append({
        "backbone": backbone,
        "recipe": recipe["tag"],
        "optimizer": "AdamW",
        "imgsz": IMGSZ,
        **{key: effective[key] for key in (
            "epochs", "patience", "batch", "lr0", "lrf",
            "cos_lr", "warmup_epochs", "weight_decay",
        )},
    })
display(pd.DataFrame(recipe_rows).set_index("backbone"))

## Build the three arms

All configurations resolve to scale `n`. The smoke test records parameter counts and
checks that every detector exposes the same P3/P4/P5 strides `(8, 16, 32)`. It does
not start training.

In [ ]:
def build_arm(backbone: str) -> YOLO:
    if backbone not in ARMS:
        raise KeyError(f"Unknown arm {backbone!r}; choose from {tuple(ARMS)}")
    timm_backbone.register()
    return YOLO(str(ARMS[backbone]), task="detect")


arm_info = []
for backbone in ARMS:
    model = build_arm(backbone)
    net = model.model
    strides = tuple(int(value) for value in net.stride)
    if strides != (8, 16, 32):
        raise RuntimeError(f"{backbone} produced unexpected strides {strides}")
    arm_info.append({
        "backbone": backbone,
        "yaml": str(ARMS[backbone]),
        "parameters_M": sum(p.numel() for p in net.parameters()) / 1e6,
        "trainable_M": sum(p.numel() for p in net.parameters() if p.requires_grad) / 1e6,
        "strides": strides,
    })
    del model, net
    gc.collect()

pd.DataFrame(arm_info).set_index("backbone")

## Train 3 backbones × 3 seeds

A completed run is skipped, so the cell is safe to restart after interruption. An
incomplete run directory is reused and trained again from its YAML initialization.
The two stock controls reuse the already completed `...backbone_stock...1024...` runs.
The tuned PVT tag creates fresh checkpoints, so the earlier `lr0=0.01` PVT run can
never be mistaken for this convergence experiment. Each run tries CUDA first and is
rebuilt on CPU only if CUDA reports an out-of-memory error.

In [ ]:
def run_name(backbone: str, seed: int) -> str:
    tag = RECIPES[backbone]["tag"]
    return f"yolov8_backbone_{tag}_adamw_cb55_{IMGSZ}_{backbone}_seed{seed}"


for backbone in ARMS:
    for seed in SEEDS:
        name = run_name(backbone, seed)
        best_weights = PROJECT / name / "weights/best.pt"
        if best_weights.is_file():
            print(f"SKIP {backbone=} {seed=}: {best_weights}")
            continue

        print(f"\nTRAIN {backbone=} {seed=} -> {name}")
        model = build_arm(backbone)
        train_args = RECIPES[backbone]["train_args"]
        shared_args = dict(
            data=DATA, optimizer="AdamW", imgsz=IMGSZ, seed=seed,
            project=str(PROJECT), name=name, exist_ok=True, **train_args,
        )
        try:
            model.train(**shared_args)
        except RuntimeError as error:
            message = str(error).lower()
            if "out of memory" not in message or "cuda" not in message:
                raise
            print(f"CUDA OOM for {backbone=} {seed=}; rebuilding on CPU")
            del model
            gc.collect()
            torch.cuda.empty_cache()
            model = build_arm(backbone)
            model.train(**shared_args, device="cpu")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## Evaluate every best checkpoint on the test split

Each of the nine selected checkpoints is evaluated once on the untouched CB55 test
split. Raw per-seed metrics are saved incrementally so a later evaluation failure does
not discard earlier rows.

In [ ]:
records = []
for backbone in ARMS:
    for seed in SEEDS:
        name = run_name(backbone, seed)
        best_weights = PROJECT / name / "weights/best.pt"
        if not best_weights.is_file():
            raise FileNotFoundError(f"Missing completed run: {best_weights}")

        print(f"\nTEST {backbone=} {seed=}")
        timm_backbone.register()
        model = YOLO(str(best_weights), task="detect")
        metrics = model.val(
            data=DATA,
            split="test",
            imgsz=IMGSZ,
            project=str(PROJECT / "test"),
            name=name,
            exist_ok=True,
            plots=False,
        )
        box = metrics.box
        records.append({
            "backbone": backbone,
            "recipe": RECIPES[backbone]["tag"],
            "seed": seed,
            "mAP50": float(box.map50),
            "mAP50_95": float(box.map),
            "precision": float(box.mp),
            "recall": float(box.mr),
            "weights": str(best_weights),
        })
        pd.DataFrame(records).to_csv(RESULTS_CSV, index=False)

        del model, metrics
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

raw_results = pd.DataFrame(records).sort_values(["backbone", "seed"])
display(raw_results.drop(columns="weights"))
print("saved ->", RESULTS_CSV)

## Aggregate comparison

Report mean and sample standard deviation across the three paired seeds. The raw CSV
above remains the source of truth for statistical tests or plots.

In [ ]:
metric_columns = ["mAP50", "mAP50_95", "precision", "recall"]
summary = raw_results.groupby("backbone")[metric_columns].agg(["mean", "std"])
summary = summary.sort_values(("mAP50_95", "mean"), ascending=False)
display(summary.round(4))

pretty = pd.DataFrame(index=summary.index)
for metric in metric_columns:
    pretty[metric] = summary.apply(
        lambda row: f"{row[(metric, 'mean')]:.4f} ± {row[(metric, 'std')]:.4f}", axis=1
    )
pretty